# Lecture 3: Behavioral Analysis

Standard benchmarks can overestimate model competence. A model may achieve high accuracy on a test set by exploiting superficial patterns rather than performing the reasoning the task is designed to measure.

Two influential findings illustrate this:

- McCoy et al. (2019), "Right for the Wrong Reasons" showed that BERT-based natural language inference models rely heavily on syntactic heuristics—such as lexical overlap between premise and hypothesis—rather than genuine entailment reasoning. Models scored well on standard NLI benchmarks yet failed systematically on examples specifically constructed to violate those heuristics.

- Jia & Liang (2017), "Adversarial Examples for Evaluating Reading Comprehension Systems" demonstrated that state-of-the-art models could be fooled by appending a single distracting sentence to a passage. The added sentence was irrelevant to the correct answer but contained superficially plausible cues, and led to a sharp drop in accuracy.

These vulnerabilities are not limited to earlier models - modern LLMs exhibit strikingly similar brittleness:

- Zheng et al. (2024), "Large Language Models Are Not Robust Multiple Choice Selectors" found that LLMs are sensitive to the ordering of answer choices in multiple-choice questions. Simply permuting the option labels (e.g., swapping A and B) can change a model's prediction, revealing a selection bias toward certain positions rather than robust reasoning about the content of each option.

- Mirzadeh et al. (2024), "GSM-Symbolic: Understanding the Limitations of Mathematical Reasoning in Large Language Models" created symbolic variants of GSM8K math problems by changing only the numerical values while preserving the reasoning structure. State-of-the-art LLMs showed significant performance drops on these superficially modified problems, with high variance across different instantiations, suggesting that models may rely on pattern matching against familiar problem templates rather than performing genuine mathematical reasoning.

The common lesson: **if we only evaluate on naturalistic test sets, we cannot distinguish robust reasoning from shortcut learning.** Targeted stress tests - small, controlled perturbations that preserve the core reasoning demand while removing surface-level cues - are essential for diagnosing what a model actually knows.

### ✍ Learning goals

We will design and apply **minimal contrasting pairs** to stress test an LLM on MMLU (i.e., a multiple-choice benchmark). Each pair consists of an original problem and a minimally modified variant that changes the answer but preserves the problem structure. If a model truly learns the knowledge, it should handle versions; if it is memorizing patterns from training data, the perturbation will expose the gap.

By the end of this notebook you will be able to:
1. Construct minimal contrasting pairs that assess robustness of the model.
2. Run a model on these pairs and measure the accuracy gap between originals and perturbations.
3. Interpret the results to identify which examples are likely memorized from pre-training data or just happen to get right.

## 0️⃣ Setup

In [1]:
from IPython.display import clear_output
import plotly.io as pio

try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    pio.renderers.default = "colab"
    !git clone https://github.com/cs221m/cs221m-course.git
    %cd cs221m-course
else:
    pio.renderers.default = "plotly_mimetype+png"
    !uv sync
    !plotly_get_chrome -y

clear_output()

**Dataset:** We use the MMLU benchmark (Massive Multitask Language Understanding), one of the standard LLM evaluation benchmark. MMLU consists of multiple-choice questions across 57 subjects. We will focus on subjects that involve math reasoning. MMLU is usually considered "continaminated", i.e., some instances in MMLU test set might occur in the pre-training dataset.

We'll use few-shot prompting to help the model infer the structure of the prompt. By including 5 question-answer pairs from the same dataset in the prompt, the model has a better chance at answering the question correctly.

In [2]:
from datasets import load_dataset, concatenate_datasets

subjects = ['abstract_algebra', 'college_mathematics', 'high_school_mathematics']

N_SHOTS = 5  # number of few-shot examples from the dev set

ds_list = []
dev_examples = {}  # subject -> list of dev examples for few-shot prompting
for subj in subjects:
    subj_ds = load_dataset('cais/mmlu', subj, split='test')
    subj_ds = subj_ds.map(lambda _: {"subject": subj}, batched=False)
    ds_list.append(subj_ds)

    # Load dev split for few-shot demonstrations
    subj_dev = load_dataset('cais/mmlu', subj, split='validation')
    dev_examples[subj] = list(subj_dev.select(range(min(N_SHOTS, len(subj_dev)))))
    print(f"  {subj}: {len(subj_ds)} test questions, {len(dev_examples[subj])} dev shots")

ds = concatenate_datasets(ds_list)
print(f"\nTotal: {len(ds)} test questions across {len(subjects)} subjects")
print(f"Few-shot: {N_SHOTS} dev examples per subject")

  abstract_algebra: 100 test questions, 5 dev shots
  college_mathematics: 100 test questions, 5 dev shots
  high_school_mathematics: 270 test questions, 5 dev shots

Total: 470 test questions across 3 subjects
Few-shot: 5 dev examples per subject


**Model:** We will evaluate `microsoft/Phi-3-small-8k-instruct`, the 7B-scale that has the highest accuracy on MMLU according to [HELM](https://crfm.stanford.edu/helm/mmlu/latest/#/leaderboard).

With the stress tests, we will assess whether the model actually knows how to solve the task or just happens to get it correct.

In [ ]:
# load model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# load on GPU with float16 (saving space)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch_type = torch.float16

model_id = 'microsoft/Phi-3-small-8k-instruct'
revision = 'main'

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side='left',
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_type,
    trust_remote_code=True,
    revision=revision
)
model = model.to(device).eval()

clear_output() # clear download logs/warnings

## 1️⃣ Standard evaluation

### Few-shot prompting

Let's see how well the model performs on our dataset when we do standard evaluation: prompt the model with the question, get the model's answer, and report the model's average accuracy.

We evaluate the model using the standard **5-shot** MMLU prompt format. For each subject, we prepend 5 completed question-answer pairs from the dev set as demonstrations, followed by the test question. We compare the log-probabilities the model assigns to the four answer tokens (A, B, C, D) and pick the argmax.

In [ ]:
LABELS = [" A", " B", " C", " D"]

def format_mmlu_prompt(question, choices, subject, labels=LABELS, few_shot_examples=None):
    """Format a single MMLU question, optionally with few-shot demonstrations.

    Args:
        question: The question text.
        choices: List of 4 answer strings.
        subject: Subject name (underscored).
        few_shot_examples: Optional list of dicts with keys 'question', 'choices', 'answer'.
            If provided, each is formatted as a completed Q&A before the test question.
    """
    subject_str = subject.replace("_", " ")
    prompt = f"The following are multiple choice questions (with answers) about {subject_str}.\n\n"

    # Few-shot demonstrations
    if few_shot_examples:
        for ex in few_shot_examples:
            prompt += ex["question"] + "\n"
            for i, choice in enumerate(ex["choices"]):
                prompt += f"{labels[i]}. {choice}\n"
            prompt += f"Answer:{labels[ex['answer']]}\n\n"

    # Test question
    prompt += question + "\n"
    for i, choice in enumerate(choices):
        prompt += f"{labels[i]}. {choice}\n"
    prompt += "Answer:"
    return prompt


def predict(model, tokenizer, prompts):
    """Return the predicted answer index (0-3) for each prompt based on
    log-probabilities of the A/B/C/D tokens."""
    label_token_ids = [tokenizer.encode(label, add_special_tokens=False)[0] for label in labels]

    inputs = tokenizer(prompts, return_tensors="pt", padding='longest', padding_side='left').to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits

    # Get logits at the last token position for each sequence
    last_logits = logits[torch.arange(len(prompts)), -1]

    # Extract logits for the four answer tokens and pick the argmax
    answer_logits = last_logits[:, label_token_ids]
    predictions = answer_logits.argmax(dim=-1).cpu().tolist()
    return predictions

The following are multiple choice questions (with answers) about abstract algebra.

The cyclic subgroup of Z_24 generated by 18 has order
 A. 4
 B. 8
 C. 12
 D. 6
Answer: A

Find the order of the factor group Z_6/<3>.
 A. 2
 B. 3
 C. 6
 D. 12
Answer: B

Statement 1 | A permutation that is a product of m even permutations and n odd permutations is an even permutation if and only if n is even. Statement 2 | Every group is isomorphic to a group of permutations.
 A. True, True
 B. False, False
 C. True, False
 D. False, True
Answer: A

Find the order of the factor group (Z_4 x Z_12)/(<2> x <2>)
 A. 2
 B. 3
 C. 4
 D. 12
Answer: C

Find the maximum possible order for some element of Z_4 x Z_6.
 A. 4
 B. 6
 C. 12
 D. 24
Answer: C

Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
 A. 0
 B. 4
 C. 2
 D. 6
Answer:


Let's print an example prompt. Notice the prompt begins with 5 few-shot examples where we give the right answer to each question - this helps the model infer how to format its output, and has been shown to improve model performance. As such, it's the standard evaluation protocol for the MMLU dataset that we're working with.

In [ ]:
# Show an example prompt (with few-shot)
example_prompt = format_mmlu_prompt(
    ds[0]["question"], ds[0]["choices"], ds[0]["subject"],
    few_shot_examples=dev_examples[ds[0]["subject"]],
)
print(example_prompt)

### Evaluation

We evaluate the model by getting its prediction on each question in our dataset, one at a time.

In [ ]:
from tqdm import tqdm
from collections import defaultdict

BATCH_SIZE = 16

# Format all prompts with few-shot demonstrations
prompts = [
    format_mmlu_prompt(ex["question"], ex["choices"], ex["subject"],
                       few_shot_examples=dev_examples[ex["subject"]])
    for ex in ds
]
gold_labels = [ex["answer"] for ex in ds]
example_subjects = [ex["subject"] for ex in ds]

# Run batched inference
all_preds = []
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Evaluating"):
    batch = prompts[i : i + BATCH_SIZE]
    preds = predict(model, tokenizer, batch)
    all_preds.extend(preds)

Evaluating: 100%|██████████| 30/30 [00:10<00:00,  2.76it/s]


Overall accuracy (5-shot): 221/470 = 47.0%

  abstract_algebra: 46/100 = 46.0%
  college_mathematics: 42/100 = 42.0%
  high_school_mathematics: 133/270 = 49.3%


Let's print the overall accuracy of the model across our dataset. Since it's a small model, its accuracy isn't too impressive. Still, it's above chance for a multiple-choice question.

In [ ]:
# Overall accuracy
correct = sum(p == g for p, g in zip(all_preds, gold_labels))
accuracy = correct / len(gold_labels)
print(f"\nOverall accuracy ({N_SHOTS}-shot): {correct}/{len(gold_labels)} = {accuracy:.1%}\n")

We can also break down the model's accuracy by category. Perhaps unsurprisingly, the model performs best on high school math, and worse on abstract algebra and college math.

In [ ]:
# Per-subject accuracy
subject_correct = defaultdict(int)
subject_total = defaultdict(int)
for p, g, s in zip(all_preds, gold_labels, example_subjects):
    subject_total[s] += 1
    if p == g:
        subject_correct[s] += 1

for s in subjects:
    acc = subject_correct[s] / subject_total[s]
    print(f"  {s}: {subject_correct[s]}/{subject_total[s]} = {acc:.1%}")

Above-chance accuracy alone does not tell us *how* the model arrives at its answers. It could be mathmetically reasoning about the problems, or it could be recognizing and recalling specific questions (or close paraphrases) from its training corpus. Since MMLU is a widely-used benchmark, many of its questions are likely present in the pre-training data of modern LLMs. The stress tests that follow are designed to disentangle these two possibilities: if the model truly understands the underlying concepts, it should handle superficial perturbations; if it is relying on memorization, even minor changes will break it.

## 2️⃣ Stress testing I: input perturbations

### Changing irrelevant variables

In this section, we will swap multiple choice options, which will preserve the value the correct answer, but change it to a different option symbol. What we're testing here is whether the model is sensitive to contextual information that isn't relevant to the problem. If the model indeed solves the MMLU question correctly, it shouldn't matter which letter the correct answer corresponds to.

Adding or changing irrelevant information is often an easy stress test - we can systematically shuffle the order of responses without changing the actual content of the question. 

In [ ]:
import random

def permute_choices(choices, answer, seed=None):
    """Randomly permute the answer choices and update the gold answer index.

    Args:
        choices: List of 4 answer strings.
        answer: Integer index (0-3) of the correct answer.
        seed: Optional seed for reproducibility.

    Returns:
        (permuted_choices, new_answer): The shuffled choice list and
            the updated index of the correct answer.
    """
    rng = random.Random(seed)
    indices = list(range(len(choices)))
    rng.shuffle(indices)
    permuted_choices = [choices[i] for i in indices]
    new_answer = indices.index(answer)
    return permuted_choices, new_answer

# Demo: permute the first example
orig_choices = ds[0]["choices"]
orig_answer = ds[0]["answer"]
perm_choices, perm_answer = permute_choices(orig_choices, orig_answer, seed=0)

print("Original:")
for i, c in enumerate(orig_choices):
    marker = " <-- correct" if i == orig_answer else ""
    print(f"  {LABELS[i]}. {c}{marker}")

print(f"\nPermuted:")
for i, c in enumerate(perm_choices):
    marker = " <-- correct" if i == perm_answer else ""
    print(f"  {LABELS[i]}. {c}{marker}")

Original:
   A. 0
   B. 4 <-- correct
   C. 2
   D. 6

Permuted:
   A. 2
   B. 0
   C. 4 <-- correct
   D. 6


### Stress test

Let's evaluate our model just like we did on the original evaluation dataset - except for this time, we swap the letter associated with the correct answer. 

In [ ]:
import numpy as np

N_TRIALS = 3

# Run N trials, each with a different permutation per question
# trial_preds[t][i] = predicted answer index for trial t, question i
trial_preds = []
trial_gold = []

for trial in range(N_TRIALS):
    perm_prompts = []
    perm_gold = []
    for idx, ex in enumerate(ds):
        # Seed combines trial and question index for unique but reproducible permutations
        seed = trial * len(ds) + idx
        perm_choices, perm_answer = permute_choices(ex["choices"], ex["answer"], seed=seed)
        perm_prompts.append(format_mmlu_prompt(
            ex["question"], perm_choices, ex["subject"],
            few_shot_examples=dev_examples[ex["subject"]],
        ))
        perm_gold.append(perm_answer)

    preds = []
    for i in tqdm(range(0, len(perm_prompts), BATCH_SIZE), desc=f"Trial {trial + 1}/{N_TRIALS}"):
        batch = perm_prompts[i : i + BATCH_SIZE]
        preds.extend(predict(model, tokenizer, batch))

    trial_preds.append(preds)
    trial_gold.append(perm_gold)

Trial 3/3: 100%|██████████| 30/30 [00:10<00:00,  2.76it/s]

Original accuracy:    47.0%
Permuted accuracy:    45.3% ± 1.0%  (mean ± std over 3 trials)
Per-trial accuracies: 46.6%, 45.1%, 44.3%
Accuracy drop:        +1.7%

Per-subject comparison (original → permuted mean ± std):
  abstract_algebra: 46.0% → 46.0% ± 0.0%
  college_mathematics: 42.0% → 42.3% ± 3.4%
  high_school_mathematics: 49.3% → 46.2% ± 1.0%

Per-question robustness (correct on original + N=3 permuted trials):
  4/4 trials correct: 117 questions
  3/4 trials correct: 64 questions
  2/4 trials correct: 59 questions
  1/4 trials correct: 82 questions
  0/4 trials correct: 148 questions


Let's report the overall accuracy. Since we could shuffle the answer in different ways, we can report the average across `n` permutation trials. Indeed, the model's accuracy drops slightly, suggesting that it sometimes memorizes the right letter for the question instead of the true answer.

In [ ]:
# Compute per-trial accuracy
trial_accs = []
for t in range(N_TRIALS):
    acc = sum(p == g for p, g in zip(trial_preds[t], trial_gold[t])) / len(ds)
    trial_accs.append(acc)

print(f"Original accuracy:    {accuracy:.1%}")
print(f"Permuted accuracy:    {np.mean(trial_accs):.1%} ± {np.std(trial_accs):.1%}  (mean ± std over {N_TRIALS} trials)")
print(f"Per-trial accuracies: {', '.join(f'{a:.1%}' for a in trial_accs)}")
print(f"Accuracy drop:        {accuracy - np.mean(trial_accs):+.1%}\n")

Just as before, we could look at the accuracy for each subject. Looks like the model is least robust in the high school math setting - perhaps some of these answers were memorized.

In [ ]:
# Per-subject breakdown across trials
print("Per-subject comparison (original → permuted mean ± std):")
for s in subjects:
    orig_acc = subject_correct[s] / subject_total[s]
    subj_mask = [i for i, es in enumerate(example_subjects) if es == s]
    subj_accs = []
    for t in range(N_TRIALS):
        sc = sum(trial_preds[t][i] == trial_gold[t][i] for i in subj_mask)
        subj_accs.append(sc / len(subj_mask))
    print(f"  {s}: {orig_acc:.1%} → {np.mean(subj_accs):.1%} ± {np.std(subj_accs):.1%}")

By systematically manipulating a feature of the input, we can focus our analysis of the model's robustness to manipulations on specific questions in the evaluation dataset. For instance, we can see which questions the model gets right after re-ordering the answer choices, which questions it misses completely, and which questions it only gets right when the answer choices are ordered as in original evaluation dataset (likely memorized).

In [ ]:
# Per-question consistency: how many trials does each question survive?
# correct_counts[i] = number of trials (out of N) where question i was answered correctly
correct_counts = []
for i in range(len(ds)):
    n_correct = sum(trial_preds[t][i] == trial_gold[t][i] for t in range(N_TRIALS))
    correct_counts.append(n_correct)

# Include original prediction in the count
orig_correct = [int(p == g) for p, g in zip(all_preds, gold_labels)]

print(f"\nPer-question robustness (correct on original + N={N_TRIALS} permuted trials):")
for k in range(N_TRIALS + 1, -1, -1):
    count = sum(
        (oc + cc) == k
        for oc, cc in zip(orig_correct, correct_counts)
    )
    print(f"  {k}/{N_TRIALS + 1} trials correct: {count} questions")

**Analysis.** The change in accuracy is within the standard diviation. We can conclude that this model is mostly robust to option permutation - it does not exhibit the strong positional bias documented by Zheng et al. (2024).

### ✏ **Exercise 1**

Let's try a different perturbation on the question where the final answer stays the same, in order to further stress-test our model.

First, let's choose a manipulation to try. In the code above, the feature we manipulated was the order of the answers. What else can we change about how the order of the answers is presented?

*Hint: One idea is to change the letters next to the answers - what if we used (1)-(4) or (i)-(iv) instead of (a)-(d)? Feel free to get more creative!*

In [ ]:
LABELS = [" A", " B", " C", " D"]

def manipulate_choices(labels, choices, answer, seed=None):
    """Manipulated the multiple-choice options systematically without 
    changing the content of the final answer.

    Args:
        labels: List of labels associated with the answers (e.g., a-d)
        choices: List of 4 answer strings.
        answer: Integer index (0-3) of the correct answer.
        seed: Optional seed for reproducibility.

    Returns:
        (new_labels, new_choices, new_answer): The manipulated choices and the
            index of the new answer
    """
    ######################## YOUR CODE HERE #############################
    # ...
    #####################################################################
    return new_labels, new_choices, new_answer

# Demo: permute the first example
orig_choices = ds[0]["choices"]
orig_answer = ds[0]["answer"]
orig_labels = LABELS
new_labels, new_choices, new_answer = permute_choices(orig_labels, orig_choices, orig_answer, seed=0)

print("Original:")
for i, c in enumerate(orig_choices):
    marker = " <-- correct" if i == orig_answer else ""
    print(f"  {orig_labels[i]}. {c}{marker}")

print(f"\nPermuted:")
for i, c in enumerate(new_choices):
    marker = " <-- correct" if i == new_answer else ""
    print(f"  {new_labels[i]}. {c}{marker}")

Let's test out how robust the model is to the manipulation you came up with! Run the code below to test the model on your input perturbation. 

In [ ]:
# no need to edit this code, just run it!

N_TRIALS = 1 # feel free to run multiple trials if you'd like!

# trial_preds[t][i] = predicted answer index for trial t, question i
trial_preds = []
trial_gold = []

for trial in range(N_TRIALS):
    perm_prompts = []
    perm_gold = []
    for idx, ex in enumerate(ds):
        # Seed combines trial and question index for unique but reproducible permutations
        seed = trial * len(ds) + idx
        perm_labels, perm_choices, perm_answer = manipulate_choices(LABELS, ex["choices"], ex["answer"], seed=seed)
        perm_prompts.append(format_mmlu_prompt(
            ex["question"], 
            perm_choices, 
            ex["subject"],
            labels=perm_labels
            few_shot_examples=dev_examples[ex["subject"]],
        ))
        perm_gold.append(perm_answer)

    preds = []
    for i in tqdm(range(0, len(perm_prompts), BATCH_SIZE), desc=f"Trial {trial + 1}/{N_TRIALS}"):
        batch = perm_prompts[i : i + BATCH_SIZE]
        preds.extend(predict(model, tokenizer, batch))

    trial_preds.append(preds)
    trial_gold.append(perm_gold)

How'd the model do? Here we'll just leave the code for the overall accuracy, but you're encouraged to dig deeper!

In [ ]:
# Compute per-trial accuracy
trial_accs = []
for t in range(N_TRIALS):
    acc = sum(p == g for p, g in zip(trial_preds[t], trial_gold[t])) / len(ds)
    trial_accs.append(acc)

print(f"Original accuracy:    {accuracy:.1%}")
print(f"Permuted accuracy:    {np.mean(trial_accs):.1%} ± {np.std(trial_accs):.1%}  (mean ± std over {N_TRIALS} trials)")
print(f"Per-trial accuracies: {', '.join(f'{a:.1%}' for a in trial_accs)}")
print(f"Accuracy drop:        {accuracy - np.mean(trial_accs):+.1%}\n")

Time to report our analysis! What perturbation did you try? How did it change the question, and (crucially) did it change the content of the answer? Finally, was the model sensitive to your perturbation, or robust enough to solve the problem in this new setting?

> FILL IN YOUR ANSWER HERE

## 3️⃣ Stress testing II: primitive substitutions

### Changing causally important variables

In the previous stress test, we tried to change input features that serve as the "background", i.e., changing their values should not change the label.

This time, we'll try perturbing input features that have **causal effects** on the label. Here's the minimal change we'll make: 

1. Take questions the model answered correctly on the original test set, 
2. Substitute a number in the problem, 
3. Recompute the correct answer (and distractor options), and 
4. Check whether the model still gets it right. 

By changing only one number, we assume the new instance is still an instance of the same "task". If the model truly knows how to solve the task, it should handle the new numbers; if it is pattern-matching against memorized question-answer pairs, it will fail.

Since recomputing the answer requires understanding each question's logic, we'll manually craft the variants for the first 5 correctly-answered examples.

In [ ]:
import re

# Find the first 5 correct examples from high school mathematics that contain at least one number.
# Our perturbation will be targeting numbers.
correct_indices_with_numbers = [
    i for i, (p, g) in enumerate(zip(all_preds, gold_labels))
    if (p == g and
        len(re.findall(r'\d+', ds[i]['question'])) > 1 and
        ds[i]['subject'] == 'high_school_mathematics')
]
target_indices = correct_indices_with_numbers[:5]

print(f"First 5 correct examples with numbers (indices: {target_indices}):\n")
for rank, idx in enumerate(target_indices):
    ex = ds[idx]
    print(f"--- Example {rank + 1} (dataset index {idx}, subject: {ex['subject']}) ---")
    print(f"Q: {ex['question']}")
    for i, c in enumerate(ex['choices']):
        marker = " <-- correct" if i == ex['answer'] else ""
        print(f"  {LABELS[i]}. {c}{marker}")
    print(f"Model predicted: {LABELS[all_preds[idx]]}")
    print()

First 5 correct examples with numbers (indices: [200, 207, 208, 215, 225]):

--- Example 1 (dataset index 200, subject: high_school_mathematics) ---
Q: If a pentagon P with vertices at (– 2, – 4), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across the line y = x to get a new pentagon, P’, then one of the vertices of P’ is
   A. (0, – 3)
   B. (4, 1)
   C. (2, 2)
   D. (– 4, –2) <-- correct
Model predicted:  D

--- Example 2 (dataset index 207, subject: high_school_mathematics) ---
Q: What is the sum of all positive integer values of $n$ such that $n^2$ is a factor of $1200$?
   A. 42 <-- correct
   B. 12
   C. 36
   D. 39
Model predicted:  A

--- Example 3 (dataset index 208, subject: high_school_mathematics) ---
Q: We roll a fair 6-sided die 5 times. What is the probability that we get a 6 in at most 2 of the rolls?
   A. \frac{125}{648}
   B. \frac{25}{648}
   C. \frac{625}{648} <-- correct
   D. \frac{1}{648}
Model predicted:  C

--- Example 4 (dataset index 215, subject: hig

### One question at a time

For each of the 5 examples above, we'll define a **generator function** that takes a trial index and returns a `(question, choices, answer)` tuple with one number substituted and the correct answer recomputed. Each generator produces `N` distinct variants. Inspect the examples printed above and fill in the generators below accordingly.

**How to write a generator:**
1. Identify a number in the question that, when changed, changes the correct answer.
2. Write a function that substitutes that number (using the trial index to pick different values) and recomputes the answer and distractors.
3. Keep the question structure, wording, and distractor style identical — only the number and the dependent answer/choices change.

In [ ]:
import math
from fractions import Fraction
from math import comb

N_VARIANTS = 5
variant_generators = {}


# --- Example 1 (idx 200): Reflection across y = x ---
# Reflecting (x, y) across y = x gives (y, x).
# We vary the first vertex of the pentagon; the correct answer is its reflection.
def gen_example_200(trial):
    first_vertices = [(-3, -5), (-1, -3), (-5, -2), (-2, -6), (-6, -1)]
    v = first_vertices[trial]
    correct = (v[1], v[0])  # reflection across y = x

    # Distractors: common reflection mistakes
    d1 = (-v[0], -v[1])  # reflection through origin
    d2 = (-v[0], v[1])   # reflection across y-axis
    d3 = (v[0], -v[1])   # reflection across x-axis

    def fmt(p):
        return f"({p[0]}, {p[1]})"

    question = (
        f"If a pentagon P with vertices at {fmt(v)}, (– 4, 1), (–1, 4), "
        f"(2, 4), and (3, 0) is reflected across the line y = x to get a "
        f"new pentagon, P', then one of the vertices of P' is"
    )

    answer_pos = trial % 4
    options = [fmt(d1), fmt(d2), fmt(d3)]
    options.insert(answer_pos, fmt(correct))
    return question, options, answer_pos

variant_generators[200] = gen_example_200


# --- Example 2 (idx 207): Sum of n where n^2 | N ---
# We vary the target number N and recompute which n satisfy n^2 | N.
def gen_example_207(trial):
    targets = [1800, 2400, 3600, 4800, 7200]
    N = targets[trial]

    # Find all positive n where n^2 divides N
    ns = [n for n in range(1, int(math.isqrt(N)) + 1) if N % (n * n) == 0]
    correct_sum = sum(ns)

    question = (
        f"What is the sum of all positive integer values of $n$ "
        f"such that $n^2$ is a factor of ${N}$?"
    )

    # Distractors: plausible nearby sums
    d1 = correct_sum + 6
    d2 = correct_sum - 3 if correct_sum > 3 else correct_sum + 9
    d3 = sum(ns[:-1]) if len(ns) > 1 else correct_sum + 12  # missing largest n
    # Ensure all four options are distinct
    for lst in [d1, d2, d3]:
        while d3 in (correct_sum, d1, d2):
            d3 += 1

    answer_pos = trial % 4
    options = [str(d1), str(d2), str(d3)]
    options.insert(answer_pos, str(correct_sum))
    return question, options, answer_pos

variant_generators[207] = gen_example_207


# --- Example 3 (idx 208): Rolling a die, P(at most k sixes) ---
# We vary the number of rolls. P(X <= 2) where X ~ Bin(n, 1/6).
def gen_example_208(trial):
    roll_counts = [4, 6, 7, 3, 8]
    n = roll_counts[trial]
    k = 2  # "at most 2"

    p = Fraction(1, 6)
    q = Fraction(5, 6)

    correct_prob = sum(comb(n, i) * p**i * q**(n - i) for i in range(k + 1))

    # Distractors: related but wrong probabilities
    p_exact_0 = q ** n
    p_exact_1 = comb(n, 1) * p * q ** (n - 1)
    p_exact_2 = comb(n, 2) * p**2 * q ** (n - 2)

    def fmt_frac(f):
        if f.denominator == 1:
            return str(f.numerator)
        return f"\\frac{{{f.numerator}}}{{{f.denominator}}}"

    question = (
        f"We roll a fair 6-sided die {n} times. What is the probability "
        f"that we get a 6 in at most {k} of the rolls?"
    )

    answer_pos = trial % 4
    options = [fmt_frac(p_exact_0), fmt_frac(p_exact_1), fmt_frac(p_exact_2)]
    options.insert(answer_pos, fmt_frac(correct_prob))
    return question, options, answer_pos

variant_generators[208] = gen_example_208


# --- Example 4 (idx 215): Ant distance (start → origin → end) ---
# Total distance = sqrt(x1^2+y1^2) + sqrt(x2^2+y2^2).
# We vary the start point, using coords that yield clean sqrt expressions.
def gen_example_215(trial):
    # (start, end) — chosen so distances are clean (Pythagorean triples or simple surds)
    configs = [
        ((-3, 4), (8, 6)),    # 5 + 10 = 15
        ((-6, 8), (5, 12)),   # 10 + 13 = 23
        ((-8, 15), (4, 3)),   # 17 + 5 = 22
        ((-5, 12), (8, 6)),   # 13 + 10 = 23
        ((-9, 12), (3, 4)),   # 15 + 5 = 20
    ]
    start, end = configs[trial]
    sx, sy = start
    ex, ey = end

    s_sq = sx**2 + sy**2
    e_sq = ex**2 + ey**2

    def simplify_sqrt(n):
        """Return (coeff, radicand) so that sqrt(n) = coeff * sqrt(radicand)."""
        coeff, rem = 1, n
        for p in range(2, int(math.isqrt(n)) + 1):
            while rem % (p * p) == 0:
                rem //= p * p
                coeff *= p
        return coeff, rem

    def fmt_sqrt(n):
        c, r = simplify_sqrt(n)
        if r == 1:
            return str(c)
        return f"{c}\\sqrt{{{r}}}" if c > 1 else f"\\sqrt{{{r}}}"

    s_str = fmt_sqrt(s_sq)
    e_str = fmt_sqrt(e_sq)
    sc, sr = simplify_sqrt(s_sq)
    ec, er = simplify_sqrt(e_sq)

    # Build correct answer string
    if sr == 1 and er == 1:
        correct = str(sc + ec)
    elif sr == 1:
        correct = f"{sc}+{e_str}"
    elif er == 1:
        correct = f"{ec}+{s_str}"
    else:
        correct = f"{s_str}+{e_str}"

    question = (
        f"An ant crawls straight from $({sx},{sy})$ to the origin, and then "
        f"continues straight on to $({ex},{ey})$. How far does it travel?"
    )

    # Distractors
    manhattan = str(abs(sx) + abs(sy) + abs(ex) + abs(ey))
    leg1_only = s_str
    leg2_only = e_str

    answer_pos = trial % 4
    options = [manhattan, leg1_only, leg2_only]
    options.insert(answer_pos, correct)
    return question, options, answer_pos

variant_generators[215] = gen_example_215


# --- Example 5 (idx 225): Greatest odd factor of n! ---
# We vary n. The greatest odd factor of n! is n! with all factors of 2 removed.
def gen_example_225(trial):
    factorials = [6, 7, 8, 9, 10]
    n = factorials[trial]

    nfact = math.factorial(n)
    # Greatest odd factor: remove all powers of 2
    odd_part = nfact
    while odd_part % 2 == 0:
        odd_part //= 2

    product_str = "\\cdot ".join(str(i) for i in range(n, 0, -1))

    question = (
        f"The symbol ${n}!$ means ${product_str}$. "
        f"What is the greatest odd integer that is a factor of ${n}!$ ?"
    )

    # Distractors: the largest odd number <= n, odd_part // 3, odd_part // 5
    d1 = n if n % 2 == 1 else n - 1
    d2 = odd_part // 3 if odd_part % 3 == 0 else odd_part - 2
    d3 = odd_part // 5 if odd_part % 5 == 0 else odd_part + 2
    # Ensure distinctness
    dists = []
    for d in [d1, d2, d3]:
        if d != odd_part and d > 0 and d not in dists:
            dists.append(d)
    fallback = 3
    while len(dists) < 3:
        if fallback not in dists and fallback != odd_part:
            dists.append(fallback)
        fallback += 2

    answer_pos = trial % 4
    options = [str(d) for d in dists[:3]]
    options.insert(answer_pos, str(odd_part))
    return question, options, answer_pos

variant_generators[225] = gen_example_225


# Quick sanity check: print one variant per generator
print("Sanity check — one variant per example:\n")
for idx in target_indices:
    q, c, a = variant_generators[idx](0)
    print(f"idx {idx}: answer_pos={a}")
    print(f"  Q: {q[:100]}...")
    for i, ch in enumerate(c):
        marker = " <-- correct" if i == a else ""
        print(f"  {LABELS[i]}. {ch}{marker}")
    print()

Sanity check — one variant per example:

idx 200: answer_pos=0
  Q: If a pentagon P with vertices at (-3, -5), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across...
   A. (-5, -3) <-- correct
   B. (3, 5)
   C. (3, -5)
   D. (-3, 5)

idx 207: answer_pos=0
  Q: What is the sum of all positive integer values of $n$ such that $n^2$ is a factor of $1800$?...
   A. 72 <-- correct
   B. 78
   C. 69
   D. 42

idx 208: answer_pos=0
  Q: We roll a fair 6-sided die 4 times. What is the probability that we get a 6 in at most 2 of the roll...
   A. \frac{425}{432} <-- correct
   B. \frac{625}{1296}
   C. \frac{125}{324}
   D. \frac{25}{216}

idx 215: answer_pos=0
  Q: An ant crawls straight from $(-3,4)$ to the origin, and then continues straight on to $(8,6)$. How f...
   A. 15 <-- correct
   B. 21
   C. 5
   D. 10

idx 225: answer_pos=0
  Q: The symbol $6!$ means $6\cdot 5\cdot 4\cdot 3\cdot 2\cdot 1$. What is the greatest odd integer that ...
   A. 45 <-- correct
   B. 5
   C. 15
   D. 9

### Stress test

Let's see how the model does! Just as with regular evaluation, we can take our crafted inputs, pass them into the model, and score the model based on its output. Since we programmatically control the causal variable, we can run multiple trials per question to test the model's robustness.

In [ ]:
# Evaluate the model on original vs. substituted variants for each example
print(f"{'='*70}")
print(f"Premise substitution results: {N_VARIANTS} variants per example")
print(f"{'='*70}\n")

all_orig_correct = 0
all_variant_correct = 0
all_variant_total = 0

for rank, idx in enumerate(target_indices):
    ex = ds[idx]
    subj = ex["subject"]
    gen = variant_generators[idx]
    shots = dev_examples[subj]

    # Original prediction (already computed — just look it up)
    orig_pred = all_preds[idx]
    orig_gold = gold_labels[idx]
    orig_is_correct = orig_pred == orig_gold
    all_orig_correct += int(orig_is_correct)

    # Generate N variants and evaluate
    var_prompts = []
    var_golds = []
    for t in range(N_VARIANTS):
        q, c, a = gen(t)
        var_prompts.append(format_mmlu_prompt(q, c, subj, few_shot_examples=shots))
        var_golds.append(a)

    var_preds = predict(model, tokenizer, var_prompts)
    var_correct = sum(p == g for p, g in zip(var_preds, var_golds))
    all_variant_correct += var_correct
    all_variant_total += N_VARIANTS

    # Print per-example results
    print(f"--- Example {rank + 1} (idx {idx}, {subj}) ---")
    print(f"  Original: {LABELS[orig_pred]} (gold {LABELS[orig_gold]}) → {'CORRECT' if orig_is_correct else 'WRONG'}")
    print(f"  Variants: {var_correct}/{N_VARIANTS} correct")
    for t in range(N_VARIANTS):
        q, c, a = gen(t)
        status = "CORRECT" if var_preds[t] == var_golds[t] else "WRONG"
        print(f"    Trial {t}: pred={LABELS[var_preds[t]]}, gold={LABELS[var_golds[t]]} → {status}")
        # Show the substituted question (first 100 chars)
        print(f"           Q: {q[:100]}{'...' if len(q) > 100 else ''}")
    print()

Premise substitution results: 5 variants per example

--- Example 1 (idx 200, high_school_mathematics) ---
  Original:  D (gold  D) → CORRECT
  Variants: 1/5 correct
    Trial 0: pred= D, gold= A → WRONG
           Q: If a pentagon P with vertices at (-3, -5), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across...
    Trial 1: pred= D, gold= B → WRONG
           Q: If a pentagon P with vertices at (-1, -3), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across...
    Trial 2: pred= D, gold= C → WRONG
           Q: If a pentagon P with vertices at (-5, -2), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across...
    Trial 3: pred= D, gold= D → CORRECT
           Q: If a pentagon P with vertices at (-2, -6), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across...
    Trial 4: pred= D, gold= A → WRONG
           Q: If a pentagon P with vertices at (-6, -1), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across...

--- Example 2 (idx 207, high_school_mathematics) --

We printed the per-question breakdown above. We can also summarize our results across different trials.

In [ ]:
# Summary
print(f"{'='*70}")
print(f"Summary across {len(target_indices)} examples:")
print(f"  Original accuracy:  {all_orig_correct}/{len(target_indices)} = {all_orig_correct/len(target_indices):.0%}")
print(f"  Variant accuracy:   {all_variant_correct}/{all_variant_total} = {all_variant_correct/all_variant_total:.0%}")
print(f"  Accuracy drop:      {all_orig_correct/len(target_indices) - all_variant_correct/all_variant_total:+.0%}")

### ✏ **Exercise 2**

In this exercise, we'll try a different manipulation on the same question. There may have multiple causally relevant parts to the input - stress testing different parts of the input might give us an indication of how the model is solving the underlying task.

Let's zoom in to the first question that the model gets right: reflecting points across the line $y = x$. Our permutation changed the **first** coordinate in the list, but there's nothing special about that point! What if we systematically manipulated one of the other points in the list?

In [ ]:
# --- Example 1 (idx 200): Reflection across y = x ---
# Reflecting (x, y) across y = x gives (y, x).
# We vary the first vertex of the pentagon; the correct answer is its reflection.
def gen_example_200(trial):
    first_vertices = [(-3, -5), (-1, -3), (-5, -2), (-2, -6), (-6, -1)]
    v = first_vertices[trial]
    correct = (v[1], v[0])  # reflection across y = x

    # Distractors: common reflection mistakes
    d1 = (-v[0], -v[1])  # reflection through origin
    d2 = (-v[0], v[1])   # reflection across y-axis
    d3 = (v[0], -v[1])   # reflection across x-axis

    def fmt(p):
        return f"({p[0]}, {p[1]})"

    question = (
        f"If a pentagon P with vertices at {fmt(v)}, (– 4, 1), (–1, 4), "
        f"(2, 4), and (3, 0) is reflected across the line y = x to get a "
        f"new pentagon, P', then one of the vertices of P' is"
    )

    answer_pos = trial % 4
    options = [fmt(d1), fmt(d2), fmt(d3)]
    options.insert(answer_pos, fmt(correct))
    return question, options, answer_pos

Re-write the function above so that it changes the 2nd vertex instead of the first one. This should be a somewhat small change to the code above (mostly changing the `question` string). Feel free to get more creative! For example, we could use `trial` to determine the index of the point we want to edit, instead of the value of the new vertex.

In [ ]:
# --- Example 1 (idx 200): Reflection across y = x ---
# Reflecting (x, y) across y = x gives (y, x).
# This time, we vary the index of the vertex that we're changing.
def gen_example_200(trial):
    ######################### YOUR CODE HERE #########################
    # ...
    return
    ##################################################################

Now, run the code below to test the model on the new causal variable. Did it change how the model handled this problem?

In [ ]:
# Evaluate the model on original vs. substituted variants for each example
variant_generators = {200: gen_example_200}

print(f"{'='*70}")
print(f"Premise substitution results: {N_VARIANTS} variants per example")
print(f"{'='*70}\n")

all_orig_correct = 0
all_variant_correct = 0
all_variant_total = 0

for rank, idx in enumerate([200]):
    ex = ds[idx]
    subj = ex["subject"]
    gen = variant_generators[idx]
    shots = dev_examples[subj]

    # Original prediction (already computed — just look it up)
    orig_pred = all_preds[idx]
    orig_gold = gold_labels[idx]
    orig_is_correct = orig_pred == orig_gold
    all_orig_correct += int(orig_is_correct)

    # Generate N variants and evaluate
    var_prompts = []
    var_golds = []
    for t in range(N_VARIANTS):
        q, c, a = gen(t)
        var_prompts.append(format_mmlu_prompt(q, c, subj, few_shot_examples=shots))
        var_golds.append(a)

    var_preds = predict(model, tokenizer, var_prompts)
    var_correct = sum(p == g for p, g in zip(var_preds, var_golds))
    all_variant_correct += var_correct
    all_variant_total += N_VARIANTS

    # Print per-example results
    print(f"--- Example {rank + 1} (idx {idx}, {subj}) ---")
    print(f"  Original: {LABELS[orig_pred]} (gold {LABELS[orig_gold]}) → {'CORRECT' if orig_is_correct else 'WRONG'}")
    print(f"  Variants: {var_correct}/{N_VARIANTS} correct")
    for t in range(N_VARIANTS):
        q, c, a = gen(t)
        status = "CORRECT" if var_preds[t] == var_golds[t] else "WRONG"
        print(f"    Trial {t}: pred={LABELS[var_preds[t]]}, gold={LABELS[var_golds[t]]} → {status}")
        # Show the substituted question (first 100 chars)
        print(f"           Q: {q[:100]}{'...' if len(q) > 100 else ''}")
    print()

What does the model's performance on the substituted problem tell you about how it's solving the task?

*Hint: What was the causal variable we substituted, and was the model sensitive to it? How does this variable factor into solving the problem (what is its causal effect), and what does it mean for the model to be sensitive to it (or not)?*

> FILL IN YOUR ANSWER HERE

## 4️⃣ Memorization: injecting the original answer as a distractor

If the model has memorized a question-answer pair, changing the numbers should not matter - the model will still gravitate toward the *original* answer value. To test this, we re-run each variant but **replace one distractor with the original correct answer**. We then measure:

1. **How often the model picks the (new) correct answer** - the model solves the task robustly.
2. **How often the model picks the original answer** - the model likely memorized the answer to the question.
3. **How often the model picks neither** - the model is sensitive to even small manipulations and doesn't correctly solve the problem.

In [ ]:
# For each example, get the original correct answer string
original_answer_strings = {}
for idx in target_indices:
    ex = ds[idx]
    original_answer_strings[idx] = ex["choices"][ex["answer"]]

print("Original correct answer values:")
for idx in target_indices:
    print(f"  idx {idx}: {original_answer_strings[idx]}")


def inject_original_answer(variant_choices, variant_answer_pos, orig_answer_str):
    """Replace one distractor with the original answer string.

    Returns (new_choices, new_correct_pos, orig_injected_pos).
    If the original answer is already present (i.e., it equals the new correct
    answer), returns None — the variant is not useful for this probe.
    """
    choices = list(variant_choices)
    correct_val = choices[variant_answer_pos]

    # If the original answer matches the new correct answer, skip
    if orig_answer_str == correct_val:
        return None

    # Pick the first distractor slot to replace
    for i in range(4):
        if i != variant_answer_pos:
            choices[i] = orig_answer_str
            orig_pos = i
            break

    return choices, variant_answer_pos, orig_pos


# Run the memorization probe
print(f"\n{'='*70}")
print(f"Memorization probe: {N_VARIANTS} variants per example")
print(f"{'='*70}\n")

total_pick_correct = 0
total_pick_original = 0
total_pick_other = 0
total_variants = 0

for rank, idx in enumerate(target_indices):
    ex = ds[idx]
    subj = ex["subject"]
    gen = variant_generators[idx]
    shots = dev_examples[subj]
    orig_ans_str = original_answer_strings[idx]

    probe_prompts = []
    probe_correct_pos = []
    probe_orig_pos = []

    for t in range(N_VARIANTS):
        q, c, a = gen(t)
        result = inject_original_answer(c, a, orig_ans_str)
        if result is None:
            continue  # variant answer == original answer, skip
        new_choices, correct_pos, orig_pos = result
        probe_prompts.append(format_mmlu_prompt(q, new_choices, subj, few_shot_examples=shots))
        probe_correct_pos.append(correct_pos)
        probe_orig_pos.append(orig_pos)

    if not probe_prompts:
        print(f"--- Example {rank + 1} (idx {idx}) ---")
        print(f"  Skipped: all variants have the same answer as the original.\n")
        continue

    preds = predict(model, tokenizer, probe_prompts)

    n_correct = sum(p == cp for p, cp in zip(preds, probe_correct_pos))
    n_original = sum(p == op for p, op in zip(preds, probe_orig_pos))
    n_other = len(preds) - n_correct - n_original

    total_pick_correct += n_correct
    total_pick_original += n_original
    total_pick_other += n_other
    total_variants += len(preds)

    print(f"--- Example {rank + 1} (idx {idx}) ---")
    print(f"  Original answer value: {orig_ans_str}")
    print(f"  Picked NEW correct:    {n_correct}/{len(preds)}")
    print(f"  Picked ORIGINAL value: {n_original}/{len(preds)}  ← memorization signal")
    print(f"  Picked OTHER:          {n_other}/{len(preds)}")
    for t_i, (p, cp, op) in enumerate(zip(preds, probe_correct_pos, probe_orig_pos)):
        if p == op:
            tag = "⚠ ORIGINAL"
        elif p == cp:
            tag = "✓ correct"
        else:
            tag = "✗ other"
        print(f"    Variant {t_i}: pred={LABELS[p]}, correct={LABELS[cp]}, original={LABELS[op]} → {tag}")
    print()

# Summary
print(f"{'='*70}")
print(f"Summary across {len(target_indices)} examples ({total_variants} usable variants):")
print(f"  Picked NEW correct answer: {total_pick_correct}/{total_variants} = {total_pick_correct/total_variants:.0%}")
print(f"  Picked ORIGINAL answer:    {total_pick_original}/{total_variants} = {total_pick_original/total_variants:.0%}  ← memorization rate")
print(f"  Picked OTHER distractor:   {total_pick_other}/{total_variants} = {total_pick_other/total_variants:.0%}")

Original correct answer values:
  idx 200: (– 4, –2)
  idx 207: 42
  idx 208: \frac{625}{648}
  idx 215: 5+2\sqrt{13}
  idx 225: 15

Memorization probe: 5 variants per example

--- Example 1 (idx 200) ---
  Original answer value: (– 4, –2)
  Picked NEW correct:    0/5
  Picked ORIGINAL value: 1/5  ← memorization signal
  Picked OTHER:          4/5
    Variant 0: pred= D, correct= A, original= B → ✗ other
    Variant 1: pred= D, correct= B, original= A → ✗ other
    Variant 2: pred= D, correct= C, original= A → ✗ other
    Variant 3: pred= A, correct= D, original= A → ⚠ ORIGINAL
    Variant 4: pred= D, correct= A, original= B → ✗ other

--- Example 2 (idx 207) ---
  Original answer value: 42
  Picked NEW correct:    2/4
  Picked ORIGINAL value: 1/4  ← memorization signal
  Picked OTHER:          1/4
    Variant 0: pred= B, correct= A, original= B → ⚠ ORIGINAL
    Variant 1: pred= C, correct= C, original= A → ✓ correct
    Variant 2: pred= B, correct= D, original= A → ✗ other
    Variant

**Analysis.** Across all 24 usable variants, the model picks the *original* answer value 38% of the time — higher than the 25% it picks the *new* correct answer. This suggests the model is likely pattern-matching against memorized associations rather than reasoning from the modified numbers.

The most striking case is **Example 4 (idx 215)**: the model picks the original answer `5+2√13` on **all 5 out of 5 variants**, even though the start and end coordinates have changed and `5+2√13` is no longer the correct distance. The model never selects the new correct answer and never selects the other distractors — it locks onto the original value with 100% consistency. This is strong evidence that the model has memorized this specific question-answer pair from its pre-training data and is recalling the stored answer rather than computing the Euclidean distances from the new coordinates.

Combining with results from the permutating options, the model is likely memorizing the answer value as opposed to the option symbol. We will explore the mechanisms behind multiple-choice questions in later lectures, which will explain why LLMs are usually robust to label permutations.